# Chronos v20 — the staged cascade (Phase 1, notebook-only)

For every level, escalate through **Memory → BFS → Forge → LADDER**, stopping at the first
stage that returns a **replay-verified** solution. Each stage gets ~10 min. Any win is
committed so the next level chains from it. References **v19/src read-only**; all new code
is here in v20.

**Targets:** crack **ls20 L5** and **ar25 L2**. **Honest mode** is hard-wired
(`V19_STORE_SOLUTIONS=0`, `V19_CACHE_FALLBACK=0`); Memory is *verified recall*, never blind
replay. All search runs on a **forked engine** → zero scored actions.

> **Memory bank:** the cascade aggregates the v19 corpus **+ the v12/v13/v17 archive caches**
> and replay-verifies each plan on the *live* game version. For **ls20** that gives a verified
> **L0–L4** chain (from v12/v13), so it reaches the real wall — **L5** — in seconds. **ar25**
> has only L0–L1 anywhere, so **L2** is the genuine wall. Loading the correct (**latest**)
> version is essential: older version-hashes under `environment_files/<gid>/` are *different
> puzzles*, so the notebook asks `arc_agi` which version the scored engine uses.

In [ ]:
# === Setup: locate v19/src (read-only), honest mode, imports ===
import os, sys, glob, re, time, json, pickle, zlib, copy, hashlib, random
from collections import deque
import numpy as np

os.environ["V19_STORE_SOLUTIONS"] = "0"
os.environ["V19_CACHE_FALLBACK"]  = "0"

def _locate_v19_src():
    here = os.getcwd()
    cands = [os.path.join(here, "..", "..", "v19", "src"),
             os.path.join(here, "..", "v19", "src")]
    for base in [os.environ.get("V19_DIR", ""),
                 "/Users/shreyas/gitrepos/OpenSource/kaggle/arc3/CommunitySolutions/chronos_solver/v19",
                 "/workspace/arc3/CommunitySolutions/chronos_solver/v19",
                 "/kaggle/working/arc3/CommunitySolutions/chronos_solver/v19"]:
        if base:
            cands += [os.path.join(base, "src"), base]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "combined_agent.py")):
            return os.path.abspath(c)
    for root in [here, os.path.dirname(here), "/workspace", "/kaggle", os.path.expanduser("~")]:
        if root and os.path.isdir(root):
            hits = glob.glob(os.path.join(root, "**", "v19", "src", "combined_agent.py"), recursive=True)
            if hits:
                return os.path.dirname(hits[0])
    raise FileNotFoundError("v19/src/combined_agent.py not found — set env V19_DIR")

V19_SRC = _locate_v19_src()
sys.path.insert(0, V19_SRC)

def _find_env_dir(start):
    d = start
    for _ in range(8):
        cand = os.path.join(d, "arc-prize-2026-arc-agi-3", "environment_files")
        if os.path.isdir(cand):
            return cand
        d = os.path.dirname(d)
    for base in [os.path.dirname(start), "/workspace", "/kaggle/input", os.path.expanduser("~")]:
        if base and os.path.isdir(base):
            hits = glob.glob(os.path.join(base, "**", "environment_files"), recursive=True)
            if hits:
                return hits[0]
    return os.path.join(start, "..", "..", "..", "..", "arc-prize-2026-arc-agi-3", "environment_files")

ENV_DIR  = _find_env_dir(V19_SRC)
CORPUS   = os.path.join(V19_SRC, "solutions")                              # v19 stored plans
ARCHIVE  = os.path.normpath(os.path.join(V19_SRC, "..", "..", "archive"))  # v12/v13/v17 banks
AGENTS   = os.path.normpath(os.path.join(ENV_DIR, "..", "ARC-AGI-3-Agents"))
FORGE_W  = next((p for p in [os.path.join(V19_SRC, "pretrained_weights.pt"),
                             "/kaggle/input/forge-pretrained-weights/pretrained_weights.pt"]
                 if os.path.exists(p)), None)

from combined_agent import BFSSolver
try:
    from combined_agent import ActionInput, GameAction
except Exception:
    from arcengine import GameAction
    try:    from arcengine import ActionInput
    except Exception: from combined_agent import ActionInput

import logging
logging.getLogger("combined_agent").setLevel(logging.ERROR)   # quiet the v19 desync chatter

random.seed(0); np.random.seed(0)
print("v19/src :", V19_SRC)
print("env_files:", ENV_DIR, "| exists:", os.path.isdir(ENV_DIR))
print("corpus   :", CORPUS, "| forge weights:", "loaded" if FORGE_W else "COLD (none found)")

## Engine driver + shared helpers
A thin white-box wrapper over the shipped game class (same calls the agent makes).
`chained_start` builds a level's true start by replaying the accepted prior-level plans;
`verify` replays prior plans + a candidate and checks the engine's `levels_completed`.

In [ ]:
_VER = {}
def live_version_dir(gid):
    # CRITICAL: load the SAME game version the live/scored engine uses. Multiple version
    # hashes can sit under environment_files/<gid>/; arc_agi picks the *latest*. Picking
    # the wrong one means solving a different puzzle (and archived plans won't transfer).
    if gid in _VER:
        return _VER[gid]
    ld = None
    try:
        if AGENTS not in sys.path:
            sys.path.insert(0, AGENTS)
        import arc_agi
        arc = arc_agi.Arcade(environments_dir=ENV_DIR, operation_mode=arc_agi.OperationMode.OFFLINE)
        ld = getattr(getattr(arc.make(gid, render_mode=None), "environment_info", None), "local_dir", None)
    except Exception:
        ld = None
    if not ld or not os.path.exists(os.path.join(ld, f"{gid}.py")):
        cands = sorted(glob.glob(os.path.join(ENV_DIR, gid, "*", f"{gid}.py")),
                       key=os.path.getmtime, reverse=True)            # fallback: newest on disk
        ld = os.path.dirname(cands[0]) if cands else None
    _VER[gid] = ld
    return ld

def resolve_game(gid):
    ld = live_version_dir(gid)
    src = os.path.join(ld, f"{gid}.py")
    assert os.path.exists(src), f"no live source for {gid} (looked in {ld})"
    m = re.search(r"class\s+(\w+)\s*\(\s*ARCBaseGame", open(src).read())
    return src, (m.group(1) if m else gid[0].upper() + gid[1:])

def snap(g):
    try:    return ("p", zlib.compress(pickle.dumps(g, -1), 1))
    except Exception: return ("d", copy.deepcopy(g))
def restore(s):
    k, p = s
    return pickle.loads(zlib.decompress(p)) if k == "p" else copy.deepcopy(p)
def fsig(f):      return hashlib.md5(np.ascontiguousarray(f).tobytes()).hexdigest()[:16]
def hist_sig(f):  return tuple(np.bincount(np.ascontiguousarray(f).flatten(), minlength=16).tolist())

class Engine:
    def __init__(self, gid, budget=600):
        self.gid = gid
        self.src, self.cls_name = resolve_game(gid)
        self.solver = BFSSolver(self.src, self.cls_name, bfs_timeout=budget)
        assert self.solver.load(), f"failed to load {gid}"
        self.game_cls   = self.solver.game_cls
        self.num_levels = len(self.game_cls()._levels)
    def fresh(self, level):
        g = self.game_cls(); g.set_level(level)
        g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        return g, np.array(r.frame[-1])
    def step(self, g, a, d=None):
        ai = ActionInput(id=GameAction.from_id(a), data=d) if d else ActionInput(id=GameAction.from_id(a))
        return g.perform_action(ai, raw=True)
    @staticmethod
    def frame(r):  return np.array(r.frame[-1]) if (r and r.frame) else None
    @staticmethod
    def goal(g, r, level):
        return bool(getattr(r, "levels_completed", 0) > level or
                    getattr(g, "_current_level_index", 0) > level)
    def scan(self, g, f0):
        bg = int(np.bincount(f0.flatten(), minlength=16).argmax())
        return self.solver._scan_actions(g, f0, bg)

def chained_start(eng, level, solutions):
    # TRUE chained start: replay the accepted prior-level plans directly. We avoid the
    # solver's _make_start_state because its strict _current_level_index check spuriously
    # "desyncs" when a plan ends exactly as the level completes (the index advances on the
    # next action). Populating solver.solutions still lets the BFS stage try true chaining.
    eng.solver.solutions.update(solutions)
    if level == 0 or any(li not in solutions for li in range(level)):
        return eng.fresh(level)
    g = eng.game_cls()
    g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    for li in range(level):
        for a, d in solutions[li]:
            r = eng.step(g, a, d)
    f = eng.frame(r)
    return (g, f) if f is not None else eng.fresh(level)

def verify(eng, level, full_path, solutions):
    g = eng.game_cls()
    g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
    for li in range(level):
        for a, d in solutions[li]:
            eng.step(g, a, d)
    r = None
    for a, d in full_path:
        r = eng.step(g, a, d)
    return eng.goal(g, r, level)

# smoke
_e = Engine("ls20"); print("ls20:", _e.num_levels, "levels |", _e.cls_name)

## Stage 1 — Memory  (verified recall)

In [ ]:
def load_memory_bank(gid):
    # Aggregate every remembered plan for this game: v19 corpus + the v12/v13/v17 archive
    # banks. Returns {level: [candidate_plan, ...]} (de-duped). Candidates are NOT trusted
    # blindly — stage_memory replay-verifies each on the LIVE version in chained context.
    sources = [os.path.join(CORPUS, f"{gid}.json")]
    sources += sorted(glob.glob(os.path.join(ARCHIVE, "v*", f"*bfs_cache_{gid}.json")))
    bank = {}
    for fp in sources:
        if not os.path.exists(fp):
            continue
        try:
            d = json.load(open(fp))
        except Exception:
            continue
        for k, v in d.items():
            if str(k).lstrip("-").isdigit() and isinstance(v, list) and v and isinstance(v[0], list):
                plan = [(a, dd) for a, dd in v]
                cands = bank.setdefault(int(k), [])
                if plan not in cands:
                    cands.append(plan)
    return bank

def stage_memory(eng, level, solutions, bank, budget=None):
    cands = bank.get(level) or []
    if not cands:
        return None, {"why": "MISS (no stored plan)"}
    t = time.time()
    for plan in cands:                            # try each remembered plan; first that re-verifies wins
        if verify(eng, level, plan, solutions):
            return list(plan), {"why": f"HIT (verified, {len(cands)} cand)", "actions": len(plan),
                                "secs": time.time() - t}
    return None, {"why": f"STALE ({len(cands)} cand failed replay-verify)", "secs": time.time() - t}

## Stage 2 — BFS  (genuine v13/v17 ladder)

In [ ]:
def stage_bfs(eng, level, solutions, budget=600):
    eng.solver.solutions.update(solutions)
    eng.solver.bfs_timeout = budget
    eng.solver.solutions.pop(level, None)              # force a fresh search
    prev = solutions.get(level - 1) if level > 0 else None
    t = time.time()
    sol = eng.solver.solve_level(level, prev_solution=prev, strategy="auto")
    dt = time.time() - t
    if sol and verify(eng, level, sol, solutions):
        return list(sol), {"actions": len(sol), "secs": dt}
    return None, {"secs": dt, "why": "no solution / failed verify"}

## Stage 3 — Forge  (pretrained black-box, driven on a forked engine)
The v19 `ForgeAgent` (ChangeNet) plays the wall level on a fork; on death/RESET we re-chain
to the level's start so it keeps learning *this* level online. A win is replay-verified.

In [ ]:
def stage_forge(eng, level, solutions, budget=600, weights=None):
    try:
        from forge_agent import ForgeAgent
    except Exception as e:
        return None, {"why": f"forge import failed: {e!r}"}
    forge = ForgeAgent(weights=weights); forge.reset(eng.gid)

    class _Obs: pass
    def avail(g):
        aa = getattr(g, "_available_actions", []) or []
        return tuple(a.value if hasattr(a, "value") else int(a) for a in aa)

    g, f = chained_start(eng, level, solutions)
    path, t0, deaths, steps = [], time.time(), 0, 0
    state = "NOT_FINISHED"
    while time.time() - t0 < budget:
        steps += 1
        o = _Obs()
        o.frame = np.ascontiguousarray(f).astype(np.uint8)
        o.levels_completed = level
        o.state = state
        o.available_actions = avail(g)
        aid, data = forge.act(o)
        if aid == 0:                                   # RESET -> re-chain (stay on this level)
            g, f = chained_start(eng, level, solutions); path = []; state = "NOT_FINISHED"; continue
        r = eng.step(g, aid, data)
        f2 = eng.frame(r)
        if f2 is None:
            continue
        path.append((aid, data)); f = f2
        state = str(getattr(getattr(r, "state", None), "value", getattr(r, "state", "")))
        if eng.goal(g, r, level):
            if verify(eng, level, path, solutions):
                return list(path), {"actions": len(path), "deaths": deaths,
                                    "steps": steps, "secs": time.time() - t0}
            g, f = chained_start(eng, level, solutions); path = []; state = "NOT_FINISHED"; continue
        if state == "GAME_OVER":
            deaths += 1
            g, f = chained_start(eng, level, solutions); path = []; state = "NOT_FINISHED"
    return None, {"deaths": deaths, "steps": steps, "secs": time.time() - t0}

## Stage 4 — LADDER  (variant re-root via Go-Explore + TTRL suffix-BFS)
The Stage-0 lever from `v19/notebooks/ladder_ttrl_validation.ipynb`: re-root the wall level
at archived deep landmarks, finish each with a short BFS, verify. (Known weak proposer —
this is the experimental long-shot stage.)

In [ ]:
def collect_landmarks(eng, level, solutions, n_iters=2500, chunk=4, depth_cap=400):
    g, f0 = chained_start(eng, level, solutions)
    base_snap = snap(g)
    cand = eng.scan(g, f0)
    if not cand:
        return [], cand
    archive = {hist_sig(f0): [0, base_snap, [], f0, 0]}
    goal_path = None
    for _ in range(n_iters):
        cells = list(archive.values())
        w = np.array([(c[0] + 1.0) / (c[4] + 1.0) for c in cells], dtype=float)
        w /= w.sum()
        c = cells[int(np.random.choice(len(cells), p=w))]; c[4] += 1
        gg = restore(c[1]); path = list(c[2])
        for _ in range(chunk):
            if len(path) >= depth_cap:
                break
            a, dd = random.choice(cand)
            r = eng.step(gg, a, dd); path.append((a, dd))
            f = eng.frame(r)
            if f is None:
                break
            if eng.goal(gg, r, level):
                goal_path = list(path); break
            s = hist_sig(f); prev = archive.get(s)
            if prev is None or len(path) > prev[0]:
                archive[s] = [len(path), snap(gg), list(path), f, 0 if prev is None else prev[4]]
        if goal_path is not None:
            break
    ranked = [tuple(c[:4]) for c in sorted(archive.values(), key=lambda x: -x[0])]
    if goal_path is not None:
        ranked = [(10**9, None, goal_path, None)] + ranked
    return ranked, cand

def bfs_from(eng, start_snap, f0, level, cand, budget=45, max_states=200000):
    t0 = time.time(); q = deque([(start_snap, [])]); seen = {fsig(f0)}; states = 0
    while q and (time.time() - t0) < budget and states < max_states:
        sn, path = q.popleft()
        for a, d in cand:
            g = restore(sn); r = eng.step(g, a, d); states += 1
            f = eng.frame(r)
            if f is None:
                continue
            if eng.goal(g, r, level):
                return path + [(a, d)], states
            s = fsig(f)
            if s in seen:
                continue
            seen.add(s); q.append((snap(g), path + [(a, d)]))
    return None, states

def stage_ladder(eng, level, solutions, budget=600, n_variants=14, ge_iters=4000):
    t0 = time.time()
    ranked, cand = collect_landmarks(eng, level, solutions, n_iters=ge_iters)
    if not cand:
        return None, {"why": "no effective actions"}
    for prog, sn, path, frm in ranked:                 # exploration may have hit the goal
        if prog >= 10**8 and verify(eng, level, path, solutions):
            return list(path), {"via": "explore", "actions": len(path), "secs": time.time() - t0}
    tried = 0
    per = max(20, int((budget - (time.time() - t0)) / max(1, n_variants)))
    for prog, sn, path, frm in [x for x in ranked[:n_variants] if x[1] is not None]:
        if time.time() - t0 > budget:
            break
        tried += 1
        suffix, states = bfs_from(eng, sn, frm, level, cand, budget=per)
        if suffix is not None:
            full = path + suffix
            if verify(eng, level, full, solutions):
                return list(full), {"via": "re-root", "actions": len(full),
                                    "prefix": len(path), "suffix": len(suffix), "secs": time.time() - t0}
    return None, {"via": None, "variants_tried": tried, "secs": time.time() - t0}

## The cascade runner
`cascade_level` tries the four stages in order on their budgets, returning the first
verified plan. `play_game` walks levels 0..target, committing each win so the next level
chains from it, and records **which stage** cracked each level.

In [ ]:
STAGES = [("memory", stage_memory), ("bfs", stage_bfs), ("forge", stage_forge), ("ladder", stage_ladder)]

def cascade_level(eng, level, solutions, corpus, budgets, use=("memory","bfs","forge","ladder"), verbose=True):
    for name, fn in STAGES:
        if name not in use:
            continue
        b = budgets.get(name, 600)
        if name == "memory":
            plan, info = fn(eng, level, solutions, corpus)
        elif name == "forge":
            plan, info = fn(eng, level, solutions, budget=b, weights=FORGE_W)
        else:
            plan, info = fn(eng, level, solutions, budget=b)
        tag = info.get("why") or info.get("via") or ("%da" % info["actions"] if plan else "fail")
        if verbose:
            print(f"    [{name:6}] {'OK ' + str(len(plan)) + 'a' if plan else 'fail':<9} "
                  f"({info.get('secs',0):.0f}s) {tag}")
        if plan is not None:
            return plan, name, info
    return None, None, None

def play_game(gid, target_level, budgets, use=("memory","bfs","forge","ladder")):
    eng = Engine(gid, budget=budgets.get("bfs", 600))
    corpus = load_memory_bank(gid)
    cov = ",".join(f"L{l}:{len(corpus[l])}c" for l in sorted(corpus))
    solutions, by_stage, per_level = {}, {}, {}
    target = min(target_level, eng.num_levels - 1)
    print(f"\n===== {gid} (targeting L{target}; {eng.num_levels} levels) =====")
    print(f"  memory bank (v19+archive): {cov or 'empty'}")
    for level in range(target + 1):
        print(f"  L{level}:")
        t = time.time()
        plan, stage, info = cascade_level(eng, level, solutions, corpus, budgets, use=use)
        if plan is None:
            per_level[level] = {"solved": False, "secs": time.time() - t}
            print(f"  L{level}: WALL — no stage cracked it ({time.time()-t:.0f}s)")
            break
        solutions[level] = plan; by_stage[level] = stage
        per_level[level] = {"solved": True, "stage": stage, "actions": len(plan), "secs": time.time() - t}
        print(f"  L{level}: SOLVED by {stage.upper()} in {len(plan)} actions ({time.time()-t:.0f}s)")
    deepest = max(by_stage) if by_stage else -1
    return {"gid": gid, "num_levels": eng.num_levels, "target": target,
            "deepest_solved": deepest, "by_stage": by_stage, "per_level": per_level,
            "reached_target": deepest >= target}

## Run the cascade on ls20 (→ L5) and ar25 (→ L2)

**Budgets** are per stage, per level. Defaults give each stage ~10 min — a full run can take
**1–2 h** (BFS on deep levels dominates). For a quick smoke, drop them to ~120 s and set
`USE` to skip Forge/LADDER.

In [ ]:
BUDGETS = {"memory": 0, "bfs": 600, "forge": 600, "ladder": 600}   # ~10 min/stage
USE     = ("memory", "bfs", "forge", "ladder")
TARGETS = {"ls20": 5, "ar25": 2}

results = {}
for gid, tgt in TARGETS.items():
    results[gid] = play_game(gid, tgt, BUDGETS, use=USE)

print("\n================ SUMMARY ================")
for gid, r in results.items():
    chain = " ".join(f"L{l}:{r['by_stage'][l]}" for l in sorted(r["by_stage"]))
    flag  = "REACHED TARGET ✅" if r["reached_target"] else f"wall after L{r['deepest_solved']}"
    print(f"{gid}: deepest L{r['deepest_solved']}/{r['target']} [{flag}]  ({chain})")

## Plots — which stage cracked each level, and the cost

In [ ]:
import matplotlib.pyplot as plt
COL = {"memory": "#8888ff", "bfs": "#888", "forge": "#e9c46a", "ladder": "#2a9d8f", None: "#d33"}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

ax = axes[0]   # stacked timeline: level vs stage that solved it (per game)
gids = list(results.keys()); ypos = {g: i for i, g in enumerate(gids)}
for g in gids:
    pl = results[g]["per_level"]
    for lvl, d in pl.items():
        c = COL.get(d.get("stage"), COL[None])
        ax.barh(ypos[g], 1, left=lvl, color=c, edgecolor="white")
        ax.text(lvl + 0.5, ypos[g], f"L{lvl}", ha="center", va="center", fontsize=8, color="black")
ax.set_yticks(list(ypos.values())); ax.set_yticklabels(gids)
ax.set_xlabel("level"); ax.set_title("Which stage cracked each level")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COL[k], label=k or "WALL") for k in ("memory","bfs","forge","ladder",None)],
          loc="upper right", fontsize=8)

ax = axes[1]   # seconds spent per level
for g in gids:
    pl = results[g]["per_level"]
    lv = sorted(pl); ax.plot(lv, [pl[k]["secs"] for k in lv], "o-", label=g)
ax.set_xlabel("level"); ax.set_ylabel("seconds (cascade total)"); ax.set_yscale("log")
ax.set_title("Cascade time per level"); ax.legend()
plt.tight_layout(); plt.show()

## Verdict & migrate trigger
- **`REACHED TARGET ✅`** on ls20 L5 and ar25 L2 → Phase 1 done. **Migrate (Phase 2):** lift
  the winning cascade into a v20 agent package + Kaggle submission.
- **`wall after L<k>`** → that's the real live wall. Read the chain (`L0:memory … L4:memory
  L5:bfs?`) to see which stage is carrying. With the archive bank, ls20 memory-chains **L0–L4**
  and the contest is entirely at **L5** (BFS→Forge→LADDER, 10 min each); ar25 memory-chains
  **L0–L1** and the contest is at **L2**. Whichever stage (if any) cracks the wall is the one
  that earns its slot.
- Forge/LADDER cracking a level **BFS could not** is the signal those stages earn their slot.

No v19 code was modified. To grow memory honestly, persist any verified plan found here into
a v20 corpus and re-run — the cascade gets faster and reaches deeper each pass.